# Tune Raft Election Timeouts Until Split Votes Stop

Companion notebook for the [Raft Leader Election](https://codemia.io/courses/distributed_systems/dst_raft_leader_election) lesson on Codemia.

**What you will build**: an event driven simulation of Raft leader election. Followers wait a random election timeout, become candidates, increment the term and request votes, and each node grants at most one vote per term. You will watch split votes happen, measure how long a cluster is leaderless after a crash, and choose the timeout range from the network round trip instead of from a default.

No GPU. Standard library, `numpy` and `matplotlib` only.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import heapq
import random

import numpy as np
import matplotlib.pyplot as plt

## 1. One election, event by event

The simulation keeps a priority queue of timed events. Each node has a term, a vote for the current term and an election timer. When the timer fires, the node becomes a candidate: it increments its term, votes for itself, resets its timer and sends `RequestVote` to every peer. Messages take a random one way delay. A node that sees a higher term steps down to follower in that term. A candidate that collects a majority becomes leader and its first heartbeat stops the others from starting elections.

We return the time until a leader is elected, how many elections were started and how many terms passed.

In [ ]:
def elect(n=5, t_min=150, t_max=300, delay=(2, 10), seed=0, trace=False):
    rng = random.Random(seed)
    term = [0] * n
    voted = [None] * n
    votes = [0] * n
    timer_gen = [0] * n        # cancels stale timers
    q = []
    def arm(i, now):
        timer_gen[i] += 1
        heapq.heappush(q, (now + rng.uniform(t_min, t_max), "timeout", i, timer_gen[i]))
    def send(now, kind, src, dst, t):
        heapq.heappush(q, (now + rng.uniform(*delay), kind, src, (dst, t)))
    for i in range(n):
        arm(i, 0.0)
    elections = 0
    while q:
        now, kind, a, b = heapq.heappop(q)
        if kind == "timeout":
            i = a
            if b != timer_gen[i]:
                continue
            term[i] += 1; voted[i] = i; votes[i] = 1; elections += 1
            if trace: print("%7.1f ms  node %d times out, starts term %d" % (now, i, term[i]))
            arm(i, now)
            for j in range(n):
                if j != i:
                    send(now, "req", i, j, term[i])
        elif kind == "req":
            cand, (j, t) = a, b
            if t > term[j]:
                term[j], voted[j], votes[j] = t, None, 0
            grant = t == term[j] and voted[j] in (None, cand)
            if grant:
                voted[j] = cand
                arm(j, now)            # granting a vote resets the follower timer
            send(now, "vote", j, cand, t if grant else -term[j])
        elif kind == "vote":
            voter, (cand, t) = a, b
            if t == term[cand] and voted[cand] == cand:
                votes[cand] += 1
                if votes[cand] > n // 2:
                    if trace: print("%7.1f ms  node %d wins term %d with %d votes" % (now, cand, t, votes[cand]))
                    return now, elections, term[cand]
            elif t < 0 and -t > term[cand]:
                term[cand], voted[cand], votes[cand] = -t, None, 0
    raise RuntimeError("no leader")

elect(seed=4, trace=True)

## 2. Split votes

When two nodes time out within about one message delay of each other, they both become candidates in the same term and can split the votes so that neither reaches a majority. Both then wait for another timeout. Find a seed where that happens with a narrow timeout range.

In [ ]:
for seed in range(200):
    t, e, term_ = elect(t_min=150, t_max=160, seed=seed)
    if e >= 3:
        print("seed", seed, ": leader after %.0f ms, %d elections" % (t, e))
        elect(t_min=150, t_max=160, seed=seed, trace=True)
        break

## 3. How the timeout range matters

Run many elections for different widths of the timeout range and record the time to a leader and the share of runs in which more than one node became a candidate, the precondition for a split vote. A wider range spreads candidates apart so that the first one usually wins before the second wakes up, but a range that is too wide makes the cluster wait longer for its first candidate.

In [ ]:
def study(width, n=5, runs=400, delay=(2, 10)):
    times, multi = [], 0
    for s in range(runs):
        t, e, _ = elect(n=n, t_min=150, t_max=150 + width, delay=delay, seed=s)
        times.append(t)
        multi += e > 1
    return np.median(times), np.percentile(times, 99), multi / runs

widths = [1, 5, 10, 25, 50, 100, 150, 300, 600]
rows = [study(w) for w in widths]
for w, (med, p99, m) in zip(widths, rows):
    print("width %4d ms: median %6.1f ms, p99 %6.1f ms, more than one candidate %5.1f%%" % (w, med, p99, 100 * m))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(widths, [r[0] for r in rows], marker="o", label="median")
ax[0].plot(widths, [r[1] for r in rows], marker="o", label="p99")
ax[0].set_xscale("log"); ax[0].set_xlabel("timeout range width (ms)"); ax[0].set_ylabel("time to leader (ms)")
ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(widths, [100 * r[2] for r in rows], marker="o", color="#dd6b20")
ax[1].set_xscale("log"); ax[1].set_xlabel("timeout range width (ms)"); ax[1].set_ylabel("runs with competing candidates (%)")
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

The p99 is what an on call engineer feels after a leader crash. It is high at both ends: narrow ranges cause repeated split votes, very wide ranges make the cluster wait for a slow first timeout.

## 4. The network sets the scale

The Raft paper recommends that the broadcast time be much smaller than the election timeout. Scale the message delay up, as for a cluster spread across regions, and watch the same timeout range start to fail.

In [ ]:
for d in [(1, 3), (2, 10), (10, 40), (40, 120)]:
    med, p99, m = study(150, delay=d, runs=300)
    print("delay %-10s median %6.1f ms, p99 %6.1f ms, more than one candidate %5.1f%%" % (d, med, p99, 100 * m))

## 5. Exercises

1. Change `n` to 3 and to 9. Does a larger cluster split votes more or less often at the same width?
2. Add **pre vote**: before incrementing its term, a candidate asks peers whether they would vote for it, and only proceeds with a majority of yes answers. Simulate a node that is partitioned for 5 seconds and rejoins, with and without pre vote, and count how many term bumps it causes.
3. Pick a timeout range for a cross region cluster with 80 ms round trips so that the p99 time to leader stays below one second.